# Alignment and Optimal Transport

Phase five. Phase four established that latent merging matches weight averaging when the
parents share a basin, and that both collapse when they do not. This notebook asks *why*
the second case fails and whether either kind of optimal transport fixes it.

Two things called OT appear here, and only one of them is about permutations.

1. **The paper's OT** (LS-Merge §3.3): a per-layer Gaussian/Bures map
   `T(z) = μ_t + A(z − μ_s)`. A single affine whitening-and-recolouring applied to every
   chunk. It repairs *support mismatch* between separately trained VAEs and different
   architectures, which is the problem the paper has.
2. **Unit-level OT**: a discrete transport plan between the filters of two models:
   Hungarian or Sinkhorn on a cost matrix. This is OTFusion (Singh & Jaggi, 2020) and the
   Wasserstein-barycentre fusion of Akash et al. (2022); Git Re-Basin is the
   hard-assignment case.

The structural reason (1) cannot do the job of (2): write a layer's latents as `Z ∈ R^{n×d}`.
The Bures map is `Z ↦ (Z − μ_s)Aᵀ + μ_t` — right multiplication, acting on coordinates.
A permutation is `Z ↦ PZ` — left multiplication, acting on the set of units. Different
sides of the matrix; no choice of `A` reorders rows.

We measure that rather than assert it.

In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
os.chdir(ROOT)

import torch
import warnings
import numpy as np
from utils import chunking as C
from tqdm import tqdm
from pathlib import Path
from utils.resnet20 import resnet20
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from utils.transformer_vae import TVAE
from utils.dataset import ResZoo
from utils.cifar_eval_data import load_cifar_eval, make_recal_loader
from utils.merging import merge_latent, merge_weight_space, scaffold
from utils.reconstruction import (interp_to_state_dict, eval_repeats, reconstruct_model,
                                  layer_rel_errors, recalibrate_bn, evaluate_range)
from utils import permutation as PM
from utils import alignment as AL
from utils.constants import (SEED, device, IMG_PATH, ZOO_PATH, ZOO_INDEPENDENT_PATH,
                             MODELS_PATH, MODELS_INDEPENDENT_PATH, BN_RECAL_REPEATS)
warnings.filterwarnings("ignore")

In [ ]:
(train_dataset_cifar, test_dataset_cifar, full_test_loader,
 zoo_train_feed, zoo_test_feed) = load_cifar_eval('./res_data/')

recal_all = make_recal_loader(train_dataset_cifar)

BETA = 3e-6
tvae_v2 = TVAE(input_dim=144, model_dim=256, latent_dim=144)
tvae_v2.load_state_dict(
    torch.load(f'{MODELS_PATH}/tvae_v2_r1_beta{BETA}.pt', weights_only=True)['state_dict'])
tvae_v2 = tvae_v2.to(device)

'''two pairs. indepfull_* differ only in the seed, which is the permutation regime the
literature is about. indep_* differ in init AND data, so a failure there is ambiguous'''
PAIRS = {
    'same-data (seed only)': ('indepfull_0', 'indepfull_1', (0, 100), (0, 100)),
    'disjoint halves':       ('indep_0', 'indep_1', (0, 50), (50, 100)),
}

CKS = {}
for label, (na, nb_, ra, rb) in PAIRS.items():
    CKS[label] = (
        torch.load(f'{MODELS_INDEPENDENT_PATH}/{na}.pt', map_location='cpu')['state_dict'],
        torch.load(f'{MODELS_INDEPENDENT_PATH}/{nb_}.pt', map_location='cpu')['state_dict'],
    )
    print(f'{label:<24} {na} / {nb_}')

MERGE_KEYS = [lm.key for lm in ResZoo(root_dir=ZOO_INDEPENDENT_PATH,
                                      model_ids=['indep_0']).meta_list[0].layers]

## 1. The permutation group of this network

Everything below produces a permutation and applies it, so the group has to be right
first. A permutation that is *almost* valid gives a model that loads without complaint and
predicts noise, and no downstream number would tell you which of the two it was.

Option-A shortcuts constrain the group. `F.pad(x[:,:,::2,::2], (0,0,0,0,pad,pad))` places
the previous stage's channels at positions `pad … pad+C_prev−1` of the new residual stream
and zeros elsewhere, so the stream permutation `R_s` must map that middle block to itself
and act on it exactly as `R_{s−1}`. Only the outer `2·pad` positions are free.

Free variables: the stage-1 stream (16), the outer positions of stages 2 and 3 (16, 32),
and each block's inner channel (16×3, 32×3, 64×3) — twelve permutations, largest 64 wide.

The gate is functional identity: apply a random valid permutation and the logits must not
move beyond floating-point noise.

In [ ]:
sd_a, sd_b = CKS['same-data (seed only)']

'''two checks, because "the logits match" is hardware dependent.

the strict one runs in float64 on the cpu, where an exact symmetry agrees to ~1e-15 and a
wrong plan is off by order 1 - no ambiguity. the second repeats it on this device and
dtype and reports a RELATIVE difference: on Ampere and later, torch uses TF32 for
convolutions by default, keeping only 10 mantissa bits, and a permuted network accumulates
over input channels in a different order. differences of ~1e-3 relative there are
accumulation order, not a broken symmetry, which is why the strict gate is the one that
decides'''
res = PM.verify_symmetry(lambda: resnet20(num_classes=100), sd_a,
                         full_test_loader, device, n_batches=4, seeds=(0, 1, 2))

'''and accuracy must be untouched'''
m0 = resnet20(num_classes=100).to(device); m0.load_state_dict(sd_a); m0.eval()
m1 = resnet20(num_classes=100).to(device)
m1.load_state_dict(PM.apply_perm(sd_a, PM.random_perm(0))); m1.eval()
a0 = evaluate_range(m0, full_test_loader, 0, 100)[2]
a1 = evaluate_range(m1, full_test_loader, 0, 100)[2]
print(f'\naccuracy before {a0:.2f}%  after a random permutation {a1:.2f}%  '
      f'(delta {a1 - a0:+.3f})')
'''not an exact equality: under TF32 a sample sitting on a decision boundary can flip.
the float64 gate above is what proves the symmetry'''
assert abs(a0 - a1) < 0.05, 'a valid permutation must not move accuracy'

'''a permutation that solves the streams independently is NOT a symmetry here'''
bad = PM.random_perm(0)
bad.res[1] = np.random.default_rng(0).permutation(32)
try:
    PM.check_valid(bad); print('FAIL: invalid permutation accepted')
except AssertionError as e:
    print(f'invalid permutation rejected: {str(e)[:90]}...')

## 2. Does the paper's OT do anything here?

Prediction: for two models of the **same architecture** trained with the **same recipe**,
and with weights z-scored per layer before encoding, the per-layer latent clouds should
already have nearly the same mean and covariance. Then `μ_s ≈ μ_t`, `Σ_s ≈ Σ_t`, so
`A ≈ I` and the Bures map is close to a no-op.

If that holds, the paper's OT is *provably inert* in this regime rather than merely
unhelpful — a much stronger statement than an argument.

We report `‖A − I‖_F / √d` per layer. A second number matters too: most layers have fewer
chunks than latent dimensions (a `layer1` conv is 16 samples in a 144-d latent), so the
Gaussian is fitted from far fewer samples than parameters and needs shrinkage. That is a
caveat about applying this OT at small scale, and it is worth reporting.

In [ ]:
def bures_table(sd_x, sd_y, label, shrinkage=1e-3):
    mx, meta = AL.encode_state_dict(tvae_v2, sd_x, device)
    my, _ = AL.encode_state_dict(tvae_v2, sd_y, device)
    Lx, Ly = AL.layer_latents(mx, meta), AL.layer_latents(my, meta)

    print(f'--- {label} ---')
    print(f'{"layer":<26}{"n":>5}{"d":>5}{"rank":>6}{"||A-I||/sqrt(d)":>18}'
          f'{"rel mean shift":>16}')
    rows = []
    for lm in meta.layers:
        r = AL.bures_report(Lx[lm.key], Ly[lm.key], shrinkage=shrinkage)
        rows.append((lm.key, lm.depth_index, r))
        print(f'{lm.key:<26}{r["n_samples"]:>5}{r["d"]:>5}{r["rank"]:>6}'
              f'{r["dev_from_identity"]:>18.4f}{r["mean_shift_rel"]:>16.4f}')
    dev = np.mean([r['dev_from_identity'] for _, _, r in rows])
    ms = np.mean([r['mean_shift_rel'] for _, _, r in rows])
    print(f'{"mean":<26}{"":>16}{dev:>18.4f}{ms:>16.4f}\n')
    return rows


bures_rows = {label: bures_table(*CKS[label], label) for label in PAIRS}

In [ ]:
'''reference points for reading ||A - I||, and the control that isolates the structural
claim.

a PURE reordering of the rows leaves the cloud identical as a set, so mu and Sigma are
unchanged and the map is EXACTLY the identity. that is the left-vs-right multiplication
argument, measured: an affine map on coordinates has no way to express a reordering of
units, so it also has nothing to gain from one.

a real network permutation is not a pure reordering of the latent rows, though. a chunk
is a [c_in, k, k] block laid out in input-channel order, so permuting the layer below
reorders values INSIDE each chunk and the cloud genuinely moves. the number to compare it
against is the other seed'''
sd_x, sd_y = CKS['same-data (seed only)']
mx, meta = AL.encode_state_dict(tvae_v2, sd_x, device)
mp, _ = AL.encode_state_dict(tvae_v2, PM.apply_perm(sd_x, PM.random_perm(0)), device)
Lx, Lp = AL.layer_latents(mx, meta), AL.layer_latents(mp, meta)

key = 'layer3.1.conv1.weight'
rng = np.random.default_rng(0)
shuffled = Lx[key][rng.permutation(len(Lx[key]))]

self_r = AL.bures_report(Lx[key], Lx[key])
shuf_r = AL.bures_report(Lx[key], shuffled)
perm_r = AL.bures_report(Lx[key], Lp[key])
pair_r = [r for k, _, r in bures_rows['same-data (seed only)'] if k == key][0]

print(f'{"comparison":<44}{"||A-I||/sqrt(d)":>18}')
print('-' * 62)
print(f'{"model vs itself":<44}{self_r["dev_from_identity"]:>18.4f}')
print(f'{"model vs its own latents, rows shuffled":<44}{shuf_r["dev_from_identity"]:>18.4f}')
print(f'{"model vs a permuted copy of itself":<44}{perm_r["dev_from_identity"]:>18.4f}')
print(f'{"model vs the other seed":<44}{pair_r["dev_from_identity"]:>18.4f}')
print()
print('row 2: a pure reordering of units is INVISIBLE to the Bures map - exactly zero.')
print('       an affine map on coordinates cannot express it and cannot undo it.')
print('rows 3 and 4: compare them. row 3 is a model FUNCTIONALLY IDENTICAL to the first,')
print('       row 4 a genuinely different one. if the two numbers are close, the statistic')
print('       the paper\'s OT operates on cannot tell a relabelling from a real difference,')
print('       so applying the map neither detects nor undoes the misalignment.')

In [ ]:
'''and the decisive check: does applying the paper's OT before interpolating change
the merged model at all? run the sweep with and without it'''
def merge_weight_with_optional_bures(sd_x, sd_y, lam, use_bures):
    w = [1.0 - lam, lam]
    if not use_bures:
        sd = merge_weight_space([sd_x, sd_y], MERGE_KEYS, w)
        return scaffold(sd, [sd_x, sd_y], MERGE_KEYS, policy='interpolate', weights=w)

    '''transport y's latents onto x's per-layer Gaussian, then decode and interpolate'''
    my, meta = AL.encode_state_dict(tvae_v2, sd_y, device)
    mx, _ = AL.encode_state_dict(tvae_v2, sd_x, device)
    Lx, Ly = AL.layer_latents(mx, meta), AL.layer_latents(my, meta)
    my_aligned = my.copy()
    for lm in meta.layers:
        A, mu_s, mu_t = AL.bures_map(Ly[lm.key], Lx[lm.key])
        z = AL.apply_bures(Ly[lm.key], A, mu_s, mu_t).numpy()
        my_aligned[lm.chunk_start:lm.chunk_end] = z
    zmix = (1 - lam) * mx + lam * my_aligned
    dec = _decode_chunks(tvae_v2, zmix, meta)
    sd = C.reconstruct_state_dict(dec, meta, sd_x)
    return scaffold(sd, [sd_x, sd_y], MERGE_KEYS, policy='interpolate', weights=w)


@torch.no_grad()
def _decode_chunks(tvae, chunk_codes, meta, batch_size=64):
    '''decode chunk-level latents back to chunks, mirroring encode_state_dict'''
    T = meta.tokens_per_seq
    z = torch.as_tensor(chunk_codes, dtype=torch.float32).view(-1, T, tvae.latent_dim)
    n_seq = z.shape[0]
    depth = torch.zeros(n_seq, dtype=torch.long)
    stage = torch.zeros(n_seq, dtype=torch.long)
    seq = torch.zeros(n_seq, dtype=torch.long)
    for lm in meta.layers:
        for k in range(lm.n_seq):
            i = lm.seq_start + k
            depth[i], stage[i], seq[i] = lm.depth_index, lm.stage, k
    tvae.eval()
    out = []
    for i in range(0, n_seq, batch_size):
        sl = slice(i, i + batch_size)
        out.append(tvae.decode(z[sl].to(device), depth[sl].to(device),
                               stage[sl].to(device), seq[sl].to(device)).cpu())
    return torch.concat(out, 0).reshape(-1, meta.chunk_size).numpy()

In [ ]:
'''the paper's OT, applied at the midpoint of the same-data pair'''
sd_x, sd_y = CKS['same-data (seed only)']
GROUPS = [(0, 100)]

for lam in (0.0, 0.5):
    plain = merge_weight_with_optional_bures(sd_x, sd_y, lam, use_bures=False)
    r_plain = eval_repeats(plain, GROUPS, recal_all, full_test_loader)
    ot = merge_weight_with_optional_bures(sd_x, sd_y, lam, use_bures=True)
    r_ot = eval_repeats(ot, GROUPS, recal_all, full_test_loader)
    print(f'lam {lam:.1f}  weight space {r_plain["all"]:6.2f} +-{r_plain["all_std"]:.2f}   '
          f'latent + Bures OT {r_ot["all"]:6.2f} +-{r_ot["all_std"]:.2f}')

## 3. Unit-level OT: matching, not transporting coordinates

Now the OT that is actually about correspondence. Build a cost matrix between the filters
of the two models and solve a transport plan over it — Hungarian for a hard assignment,
Sinkhorn for an entropic one read out as a permutation.

Three descriptors, so that "latent vs weight" is isolated from "one-sided vs two-sided":

| mode | descriptor | sides |
|---|---|---|
| `weight_full` | raw weights, Git Re-Basin objective | filter rows **and** the columns the unit is read through |
| `weight_rows` | raw filter rows | rows only |
| `latent` | the VAE's per-filter codes | rows only |

`latent` is structurally one-sided: a filter is 1/2/4 chunk-aligned rows, but the *column*
a unit is read through cuts across chunks and has no latent code. So `weight_rows` is the
control that makes the comparison fair.

**Sanity gate before any of it means anything:** plant a known permutation and check each
mode recovers it. A filter's descriptor is not invariant to the permutation of the layer
below (it is laid out in input-channel order), so descriptors must be recomputed against
the current alignment and the match re-solved — the same coordinate descent weight
matching needs. One shot recovers well under half; iterating recovers all of it.

In [ ]:
def descriptor_fns():
    return {
        'weight_rows': AL.weight_filter_rows,
        'latent': AL.latent_descriptor_fn(tvae_v2, device),
    }


print('planted-permutation recovery (must be 100% or nothing below is interpretable)')
print(f'{"mode":<16}{"single shot":>14}{"iterated":>12}{"max |A-perm(B)|":>18}')
print('-' * 60)

sd_ref = CKS['same-data (seed only)'][0]
true = PM.random_perm(7)
sd_shuf = PM.apply_perm(sd_ref, true)
want = PM.invert(true)

got = AL.weight_match(sd_ref, sd_shuf, iters=10)
rec = PM.apply_perm(sd_shuf, got)
err = max(float((sd_ref[k].double() - rec[k].double()).abs().max())
          for k in sd_ref if sd_ref[k].dtype.is_floating_point)
print(f'{"weight_full":<16}{"-":>14}'
      f'{np.mean(list(AL.perm_agreement(got, want).values())):>11.1%}{err:>18.2e}')

for name, fn in descriptor_fns().items():
    one = AL.match_from_descriptors(fn(sd_ref), fn(sd_shuf))
    itr = AL.match_iterative(sd_ref, sd_shuf, fn, iters=10)
    rec = PM.apply_perm(sd_shuf, itr)
    err = max(float((sd_ref[k].double() - rec[k].double()).abs().max())
              for k in sd_ref if sd_ref[k].dtype.is_floating_point)
    print(f'{name:<16}{np.mean(list(AL.perm_agreement(one, want).values())):>13.1%}'
          f'{np.mean(list(AL.perm_agreement(itr, want).values())):>12.1%}{err:>18.2e}')

In [ ]:
'''Hungarian vs Sinkhorn on the same cost matrices. the entropic plan is read out as a
hard permutation: a soft plan is not a symmetry of the network, so the merged weights
would not be well defined'''
for solver in ('hungarian', 'sinkhorn'):
    got = AL.weight_match(sd_ref, sd_shuf, iters=10, solver=solver)
    print(f'  {solver:<12} planted-permutation agreement '
          f'{np.mean(list(AL.perm_agreement(got, want).values())):.1%}')

## 4. Align, then merge

The comparison the whole notebook exists for. For each pair, and each alignment method,
align model B onto model A and sweep λ.

Baselines: no alignment at all, in weight space and in latent space — the phase four
numbers. Then Git Re-Basin (`weight_full`), then unit-level OT on latent descriptors.

The question is not whether alignment helps — it should. It is whether **latent**
descriptors find a better correspondence than raw weights do. That is the strongest
testable version of LS-Merge's premise, and either answer is a result.

In [ ]:
def aligned_sweep(label, lambdas=(0.0, 0.25, 0.5, 0.75, 1.0)):
    sd_x, sd_y = CKS[label]
    lo_a, hi_a = PAIRS[label][2]
    groups = [(0, 100)]

    methods = {'none': None,
               'weight_full': 'weight_full',
               'weight_rows': 'weight_rows',
               'latent': 'latent'}
    fns = descriptor_fns()
    perms = {}
    for name in methods:
        if name == 'none':
            perms[name] = PM.identity_perm()
        elif name == 'weight_full':
            perms[name] = AL.weight_match(sd_x, sd_y, iters=10)
        else:
            perms[name] = AL.match_iterative(sd_x, sd_y, fns[name], iters=10)
        PM.check_valid(perms[name])

    print(f'=== {label} ===')
    print('permutation agreement with Git Re-Basin:')
    for name in ('weight_rows', 'latent'):
        ag = np.mean(list(AL.perm_agreement(perms[name], perms['weight_full']).values()))
        print(f'  {name:<14}{ag:.1%}')

    rows = {}
    for name, perm in perms.items():
        sd_y_aligned = PM.apply_perm(sd_y, perm)
        accs = []
        for lam in lambdas:
            w = [1.0 - lam, lam]
            sd = merge_weight_space([sd_x, sd_y_aligned], MERGE_KEYS, w)
            sd = scaffold(sd, [sd_x, sd_y_aligned], MERGE_KEYS,
                          policy='interpolate', weights=w)
            r = eval_repeats(sd, groups, recal_all, full_test_loader)
            accs.append((lam, r['all'], r['all_std']))
        rows[name] = accs
        line = '  '.join(f'{a:5.2f}' for _, a, _ in accs)
        print(f'  {name:<14}{line}')
    return rows


print(f'{"":<16}' + '  '.join(f'{l:>5}' for l in (0.0, 0.25, 0.5, 0.75, 1.0)) + '   (lambda)')
sweeps = {label: aligned_sweep(label) for label in PAIRS}

In [ ]:
'''the summary table: the midpoint is where permutation misalignment bites'''
print(f'{"pair":<24}{"alignment":<16}{"lam=0.5":>10}{"vs none":>10}')
print('-' * 60)
for label, rows in sweeps.items():
    base = [a for l, a, _ in rows['none'] if abs(l - 0.5) < 1e-9][0]
    for name, accs in rows.items():
        mid = [a for l, a, _ in accs if abs(l - 0.5) < 1e-9][0]
        print(f'{label:<24}{name:<16}{mid:>10.2f}{mid - base:>+10.2f}')

fig, axes = plt.subplots(1, len(sweeps), figsize=(5.5 * len(sweeps), 4), squeeze=False)
for ax, (label, rows) in zip(axes[0], sweeps.items()):
    for name, accs in rows.items():
        ax.plot([l for l, _, _ in accs], [a for _, a, _ in accs], marker='o', label=name)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('lambda'); ax.set_ylabel('accuracy % (all 100)')
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(IMG_PATH, 'alignment_merge.png'), dpi=150)
plt.show()

## What to take from this

Read the results in this order.

1. **Section 1** must pass, or nothing else is interpretable. A random valid permutation
   has to leave the logits and the accuracy unchanged.

2. **Section 2** decides whether the paper's OT is in play for same-architecture models.
   Three readings matter:
   - A pure reordering of the latent rows gives `‖A − I‖ = 0` exactly. That is the
     left-vs-right multiplication argument as a measurement: the Bures map cannot express
     a permutation, so it cannot undo one.
   - A real network permutation *does* move the cloud, because a chunk is laid out in
     input-channel order and permuting the layer below reorders values inside it. So the
     deviation is not zero — compare it to the other-seed row instead.
   - If those two are close, the statistic is blind to the distinction that matters: it
     reports about as much "misalignment" for a functionally identical model as for a
     genuinely different one.

   None of this refutes the paper. LS-Merge never tests same-architecture different-seed
   merging and never claims it; its OT targets support mismatch between separately trained
   VAEs and different architectures, which is a real and different problem. The
   contribution here is naming the boundary precisely and measuring it.

3. **Section 3** is the sanity gate for unit-level OT. The gap between one-shot and
   iterated matching is itself a finding: a filter's latent code is *not*
   permutation-invariant, for the same input-channel-order reason. Any future
   permutation-invariant weight encoder has to address that directly, and it is the reason
   a set-attention encoder alone would not be enough.

4. **Section 4** is the result. If `latent` matches or beats `weight_full`, the learned
   representation makes correspondence easier to find and LS-Merge's premise transfers to
   this regime. If it does not, that is a clean negative on the one setting where latent
   space should have an edge — much stronger than "both collapse".

   Read `weight_rows` before concluding anything about `latent`: latent descriptors are
   structurally one-sided, so `weight_rows` is what isolates "learned vs raw descriptor"
   from "one-sided vs two-sided".

Two honest limits of the whole exercise. Alignment is applied **in weight space** even
when the correspondence is *found* in latent space, because that is what makes the merged
model well defined — so this tests whether latent codes are better descriptors for
matching, not whether merging should happen in latent space. And the Gaussian fits in
section 2 are estimated from fewer samples than dimensions for every layer below `layer3`,
which is why shrinkage is on by default and why the rank column is reported.